# 🌾 Crop Recommendation System — 10 Features

| Step | Description |
|------|-------------|
| 1 | Install & Import Libraries |
| 2 | Upload ZIP & Extract |
| 3 | Load & Explore Dataset |
| 4 | ➕ Add 3 New Features (10 total) |
| 5 | 📊 Data Visualizations |
| 6 | 🤖 Train Random Forest Model |
| 7 | 📈 Model Evaluation |
| 8 | 🌱 Interactive Crop Predictor |

## ⚙️ Step 1: Install & Import Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn ipywidgets -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import ipywidgets as widgets
from IPython.display import display, clear_output
import zipfile, os, io, warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## 📂 Step 2: Upload ZIP & Extract

In [ ]:
from google.colab import files

# ── Upload the zip file ──────────────────────────────────────────────────────
print('📁 Please upload your zip file (crop_with_features.zip or crop_10features.zip)...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'\n✅ Uploaded: {zip_name}')

# ── Extract the zip ──────────────────────────────────────────────────────────
extract_path = '/content/crop_project'
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f'\n📦 Extracted files:')
for f in os.listdir(extract_path):
    size = os.path.getsize(os.path.join(extract_path, f))
    print(f'   ✅ {f}  ({size/1024:.1f} KB)')

## 📊 Step 3: Load & Explore Dataset

In [ ]:
# Load the original CSV
csv_path = '/content/crop_project/Crop_recommendation.csv'
df = pd.read_csv(csv_path)

print('=== Dataset Overview ===')
print(f'  Rows    : {df.shape[0]}')
print(f'  Columns : {df.shape[1]}')
print(f'  Crops   : {df["label"].nunique()} unique crops')
print(f'  Missing : {df.isnull().sum().sum()} null values')
print('\n=== Columns ===')
print(list(df.columns))
display(df.head(5))

---
## ➕ Step 4: Add 3 New Features → Total 10 Input Features

| # | New Feature | Formula | Why Useful |
|---|-------------|---------|------------|
| 8 | `NPK_ratio` | N / (P + K + 1) | Nutrient balance — key in agriculture |
| 9 | `temp_humidity_index` | temperature × humidity / 100 | Climate stress indicator |
| 10 | `soil_fertility_score` | (N + P + K) / 3 | Overall soil health score |

In [ ]:
# ── Feature Engineering ──────────────────────────────────────────────────────

# New Feature 1: NPK_ratio
# Measures nitrogen dominance relative to phosphorus & potassium
df['NPK_ratio'] = (df['N'] / (df['P'] + df['K'] + 1)).round(4)

# New Feature 2: temp_humidity_index
# Captures the combined effect of heat and moisture on crop growth
df['temp_humidity_index'] = (df['temperature'] * df['humidity'] / 100).round(4)

# New Feature 3: soil_fertility_score
# Average of all three nutrients — overall soil richness
df['soil_fertility_score'] = ((df['N'] + df['P'] + df['K']) / 3).round(4)

# Reorder columns: 10 inputs first, label last
feature_cols = ['N','P','K','temperature','humidity','ph','rainfall',
                'NPK_ratio','temp_humidity_index','soil_fertility_score']
df = df[feature_cols + ['label']]

print('✅ New features added!')
print(f'   Dataset shape : {df.shape}  (2200 rows × 11 columns)')
print(f'   Input features: {len(feature_cols)}')
print(f'   Target column : label')
print('\n=== Updated Dataset Sample ===')
display(df.head(5))
print('\n=== Statistical Summary of New Features ===')
display(df[['NPK_ratio','temp_humidity_index','soil_fertility_score']].describe().round(3))

---
## 📊 Step 5: Data Visualizations

In [ ]:
# ── 1. All 10 Feature Distributions ─────────────────────────────────────────
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()
colors = ['#4CAF50','#2196F3','#FF9800','#E91E63','#9C27B0',
          '#00BCD4','#FF5722','#795548','#607D8B','#FFC107']
for i, (feat, color) in enumerate(zip(feature_cols, colors)):
    axes[i].hist(df[feat], bins=30, color=color, edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{feat}', fontweight='bold', fontsize=11)
    axes[i].axvline(df[feat].mean(), color='black', linestyle='--', linewidth=1.2,
                    label=f'Mean: {df[feat].mean():.2f}')
    axes[i].legend(fontsize=8)
for j in range(len(feature_cols), len(axes)):
    axes[j].axis('off')
plt.suptitle('📈 Distribution of All 10 Input Features', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Correlation Heatmap (10 features) ────────────────────────────────────
plt.figure(figsize=(12, 9))
corr = df[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', mask=mask,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('🔗 Correlation Heatmap — All 10 Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. New Feature: NPK_ratio by Crop ───────────────────────────────────────
plt.figure(figsize=(14, 5))
crop_npk = df.groupby('label')['NPK_ratio'].mean().sort_values(ascending=False)
bars = plt.bar(crop_npk.index, crop_npk.values,
               color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(crop_npk))),
               edgecolor='white')
plt.title('🧪 Average NPK Ratio per Crop', fontsize=14, fontweight='bold')
plt.xlabel('Crop')
plt.ylabel('NPK Ratio')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. New Feature: Soil Fertility Score by Crop ─────────────────────────────
plt.figure(figsize=(14, 5))
crop_sfs = df.groupby('label')['soil_fertility_score'].mean().sort_values(ascending=False)
plt.bar(crop_sfs.index, crop_sfs.values,
        color=plt.cm.YlGn(np.linspace(0.3, 0.95, len(crop_sfs))),
        edgecolor='white')
plt.title('🌱 Average Soil Fertility Score per Crop', fontsize=14, fontweight='bold')
plt.xlabel('Crop')
plt.ylabel('Soil Fertility Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. New Feature: Temp-Humidity Index by Crop ──────────────────────────────
plt.figure(figsize=(14, 5))
crop_thi = df.groupby('label')['temp_humidity_index'].mean().sort_values(ascending=False)
plt.bar(crop_thi.index, crop_thi.values,
        color=plt.cm.Blues(np.linspace(0.4, 0.9, len(crop_thi))),
        edgecolor='white')
plt.title('🌡️ Average Temp-Humidity Index per Crop', fontsize=14, fontweight='bold')
plt.xlabel('Crop')
plt.ylabel('Temp-Humidity Index')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 🤖 Step 6: Train Random Forest Model (10 Features)

In [ ]:
# Encode labels
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])

X = df[feature_cols]
y = df['label_enc']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f'🔀 Train samples : {X_train.shape[0]}')
print(f'🔀 Test samples  : {X_test.shape[0]}')
print(f'📐 Features used : {X_train.shape[1]}')

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'\n✅ Model trained!')
print(f'🎯 Test Accuracy : {acc*100:.2f}%')

## 📈 Step 7: Model Evaluation

In [ ]:
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Confusion Matrix
plt.figure(figsize=(16, 13))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_,
            linewidths=0.3, linecolor='white')
plt.title('📉 Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Importance (all 10) ──────────────────────────────────────────────
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

plt.figure(figsize=(9, 6))
colors_imp = ['#FF7043' if f in ['NPK_ratio','temp_humidity_index','soil_fertility_score']
              else '#42A5F5' for f in feat_imp.index]
bars = plt.barh(feat_imp.index, feat_imp.values, color=colors_imp, edgecolor='white')
plt.xlabel('Importance Score')
plt.title('🏆 Feature Importance — All 10 Features\n(🟠 New Features  🔵 Original)', fontweight='bold')
for bar, val in zip(bars, feat_imp.values):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

---
## 🌱 Step 8: Interactive Crop Predictor (10 Features)
Adjust all sliders and click **Predict** to get the best crop recommendation!

In [ ]:
style  = {'description_width': '180px'}
layout = widgets.Layout(width='450px')

# Original 7 features
w_N    = widgets.FloatSlider(value=90,  min=0,   max=140, step=1,   description='🌿 Nitrogen (N):',          style=style, layout=layout)
w_P    = widgets.FloatSlider(value=42,  min=5,   max=145, step=1,   description='🌿 Phosphorus (P):',        style=style, layout=layout)
w_K    = widgets.FloatSlider(value=43,  min=5,   max=205, step=1,   description='🌿 Potassium (K):',         style=style, layout=layout)
w_temp = widgets.FloatSlider(value=25,  min=8,   max=45,  step=0.5, description='🌡️ Temperature (°C):',     style=style, layout=layout)
w_hum  = widgets.FloatSlider(value=80,  min=14,  max=100, step=0.5, description='💧 Humidity (%):',         style=style, layout=layout)
w_ph   = widgets.FloatSlider(value=6.5, min=3.5, max=10,  step=0.1, description='🧪 pH Level:',             style=style, layout=layout)
w_rain = widgets.FloatSlider(value=200, min=20,  max=300, step=1,   description='🌧️ Rainfall (mm):',        style=style, layout=layout)

btn = widgets.Button(description='🌾 Predict Best Crop',
                     button_style='success',
                     layout=widgets.Layout(width='240px', height='42px'))
out = widgets.Output()

def on_predict(b):
    with out:
        clear_output(wait=True)
        N    = w_N.value
        P    = w_P.value
        K    = w_K.value
        temp = w_temp.value
        hum  = w_hum.value
        ph   = w_ph.value
        rain = w_rain.value

        # Compute 3 new features automatically
        npk_ratio   = round(N / (P + K + 1), 4)
        thi         = round(temp * hum / 100, 4)
        sfs         = round((N + P + K) / 3, 4)

        print(f'  📐 Auto-computed features:')
        print(f'     NPK Ratio           : {npk_ratio}')
        print(f'     Temp-Humidity Index : {thi}')
        print(f'     Soil Fertility Score: {sfs}')

        input_data = np.array([[N, P, K, temp, hum, ph, rain, npk_ratio, thi, sfs]])
        pred_enc   = model.predict(input_data)[0]
        pred_crop  = le.inverse_transform([pred_enc])[0]
        proba      = model.predict_proba(input_data)[0]
        top3_idx   = np.argsort(proba)[::-1][:3]
        top3_crops = le.inverse_transform(top3_idx)
        top3_proba = proba[top3_idx]

        print('\n' + '─'*48)
        print(f'  🌱 Recommended Crop  : {pred_crop.upper()}')
        print(f'  🎯 Confidence        : {top3_proba[0]*100:.1f}%')
        print('─'*48)
        print('  📋 Top 3 Recommendations:')
        for rank, (crop, prob) in enumerate(zip(top3_crops, top3_proba), 1):
            bar = '█' * int(prob * 30)
            print(f'  {rank}. {crop:<15} {bar} {prob*100:.1f}%')
        print('─'*48)

        # Bar chart
        fig, ax = plt.subplots(figsize=(6, 3))
        bar_colors = ['#4CAF50','#2196F3','#FF9800']
        ax.barh(top3_crops[::-1], top3_proba[::-1]*100, color=bar_colors[::-1], edgecolor='white')
        ax.set_xlabel('Confidence (%)')
        ax.set_title('Top 3 Crop Recommendations', fontweight='bold')
        for i, v in enumerate(top3_proba[::-1]*100):
            ax.text(v + 0.5, i, f'{v:.1f}%', va='center', fontsize=10)
        ax.set_xlim(0, 115)
        plt.tight_layout()
        plt.show()

btn.on_click(on_predict)

print('🎛️  Adjust the sliders and click Predict (new features are computed automatically):')
display(widgets.VBox([
    widgets.HTML('<b>── Original 7 Features ──</b>'),
    w_N, w_P, w_K, w_temp, w_hum, w_ph, w_rain,
    widgets.HTML('<b>── 3 New Features (auto-computed) ──</b>'),
    widgets.HTML('NPK_ratio, temp_humidity_index, soil_fertility_score are calculated from your inputs above.'),
    widgets.HBox([btn]),
    out
]))

---
## 💾 Step 9: Save Updated Dataset
Save the 10-feature dataset as a new CSV.

In [ ]:
save_path = '/content/Crop_recommendation_10features.csv'
df[feature_cols + ['label']].to_csv(save_path, index=False)
print(f'✅ Saved: {save_path}')
print(f'   Shape : {df.shape}  (2200 rows × 11 columns)')
print(f'   10 input features + 1 label column')

# Download it
from google.colab import files
files.download(save_path)
print('📥 Download started!')

---
## 🏁 Final Summary

| # | Feature | Type | Status |
|---|---------|------|--------|
| 1 | N (Nitrogen) | Soil Nutrient | ✅ Original |
| 2 | P (Phosphorus) | Soil Nutrient | ✅ Original |
| 3 | K (Potassium) | Soil Nutrient | ✅ Original |
| 4 | temperature | Weather | ✅ Original |
| 5 | humidity | Weather | ✅ Original |
| 6 | ph | Soil | ✅ Original |
| 7 | rainfall | Weather | ✅ Original |
| 8 | NPK_ratio | Engineered | 🆕 New |
| 9 | temp_humidity_index | Engineered | 🆕 New |
| 10 | soil_fertility_score | Engineered | 🆕 New |
| 11 | label | Target | 🎯 Output |